# Evidence-oriented clinical review (synthetic)
This offline demonstration creates synthetic M1→M4 inputs. Bands are review routing, not diagnosis, risk, pathogenicity, or actionability.

In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    previously_imported = sys.modules.get("genome_evidence")
    if previously_imported is not None:
        previous_file = Path(getattr(previously_imported, "__file__", "")).resolve()
        if not previous_file.is_relative_to(CHECKOUT.resolve()):
            raise RuntimeError(
                "genome_evidence was already imported elsewhere; restart the runtime"
            )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import validate_workspace

if PROFILE == "personal_drive":
    workspace = validate_workspace(Path(WORKSPACE_ROOT))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
import json
import shutil
import tempfile
from hashlib import sha256
from pathlib import Path

from genome_evidence.evidence import ingest_clinvar_vcv, link_external_evidence
from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run
from genome_evidence.prioritization import prioritize_clinical_variants
from genome_evidence.prioritization.models import AnalysisContext, ClinicalPrioritizationConfig

In [ ]:
root = Path(tempfile.mkdtemp(prefix="synthetic-m4-"))
source = root / "synthetic.txt"
source.write_text("# genome build: GRCh38\nsynthetic-evidence-marker\t1\t101\tAG\n")
markers = root / "markers.json"
markers.write_text(
    json.dumps(
        [
            {
                "marker_id": "synthetic-evidence-marker",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 101,
                "reference": "A",
                "alternate": "G",
                "orientation": "none",
                "orientation_authoritative": True,
            }
        ]
    )
)
fasta = root / "synthetic.fa"
fasta.write_text(">1\n" + "A" * 200 + "\n")
m1 = root / "m1"
ingest_23andme(source, m1, Ingest23andMeConfig(genome_build_override="GRCh38"))
m2 = root / "m2"
normalize_m1_run(m1, m2, NormalizationConfig(marker_definitions=markers, target_reference=fasta))
xml = root / "synthetic-clinvar.xml"
xml.write_text(
    '<ReleaseSet Dated="2026-07-01" ReleaseID="synthetic-m4">'
    '<VariationArchive Accession="VCV999000001" Version="1" RecordStatus="current">'
    '<ClassifiedRecord><SimpleAllele AlleleID="1"><SequenceLocation Assembly="GRCh38" '
    'Chr="1" positionVCF="101" referenceAlleleVCF="A" alternateAlleleVCF="G"/>'
    '</SimpleAllele><Classifications><GermlineClassification DateLastEvaluated="2025-01-01">'
    "<Description>Pathogenic</Description><ReviewStatus>criteria provided, single submitter"
    "</ReviewStatus></GermlineClassification></Classifications>"
    '<ClinicalAssertion Accession="SCV999000001" Version="1" RecordStatus="current">'
    '<Submitter Name="Fabricated Laboratory"/><GermlineClassification '
    'DateLastEvaluated="2025-01-01"><Description>Uncertain significance</Description>'
    "<ReviewStatus>criteria provided, single submitter</ReviewStatus>"
    "</GermlineClassification></ClinicalAssertion></ClassifiedRecord></VariationArchive>"
    "</ReleaseSet>"
)
evidence = root / "evidence"
ingest_clinvar_vcv(xml, evidence)
annotation = root / "annotation"
link_external_evidence(m2, evidence, annotation)
policy = root / "policy.json"
repo = Path(__import__("genome_evidence").__file__).parents[2]
shutil.copyfile(repo / "references/clinvar-germline-review-policy-v1.json", policy)
m4 = root / "m4"
result = prioritize_clinical_variants(
    m2,
    evidence,
    annotation,
    m4,
    ClinicalPrioritizationConfig(
        policy_path=policy, analysis_context=AnalysisContext.GERMLINE_CONSTITUTIONAL
    ),
)

In [ ]:
assert result.policy_identity.file_sha256 == sha256(policy.read_bytes()).hexdigest()
assert all(
    c.priority_band.value not in {"diagnosis", "risk", "actionability"} for c in result.candidates
)
assert len(result.candidates) == 1
assert result.candidates[0].priority_band.value == "review_first"
assert all(p.scv_assertion_ids and p.vcv_assertion_ids for p in result.profiles)
assert {link.source_term for link in result.candidate_assertion_links} == {
    "Pathogenic",
    "Uncertain significance",
}
assert all(link.assertion_instance_id for link in result.candidate_assertion_links)
assert all("diagnosis" not in rationale.explanation for rationale in result.rationales)
manifest = json.loads((m4 / "manifest.json").read_text())
assert all(
    sha256((m4 / name).read_bytes()).hexdigest() == digest
    for name, digest in manifest["artifacts"].items()
)
assert "not a negative genetic test" in (m4 / "prioritization_report.md").read_text()
shutil.rmtree(root)